# Preparar catálogo y consultas

Carga ESCI, filtra productos en español y separa consultas para ajustar y medir la recuperación. Las etiquetas se guardan aparte y solo se usan en las métricas.

**Entrada:** archivos Parquet en `input/` o `data/raw/`.
**Salida:** catálogo, consultas y etiquetas en `output/datos/`. Ejecuta desde la raíz del proyecto.

In [5]:
#%pip -q install -r ../requirements-lab4-modular.txt

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import json
import pandas as pd
from lab4_asistente import config as C
from lab4_asistente.datos import preparar


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Seleccionar consultas y productos

Se eligen 150 consultas `train` para desarrollo y 250 `test` para medición. La selección usa `query_id` y conserva todos los productos anotados; si hace falta, añade productos de catálogo hasta alcanzar el tamaño configurado.

In [7]:
resumen_datos = preparar(
    C.RUTA_INPUT, C.RUTA_DATOS, C.RUTA_EVAL,
    n_eval=C.N_CONSULTAS_EVAL,
    n_dev=C.N_CONSULTAS_DEV,
    min_catalogo=C.MIN_CATALOGO,
    semilla=C.SEMILLA,
)
print(json.dumps(resumen_datos, indent=2, ensure_ascii=False))


{
  "productos": 10688,
  "distractores": 0,
  "consultas_eval": 250,
  "consultas_dev": 150,
  "pares_anotados": 10928,
  "etiquetas": {
    "dev": {
      "C": 305,
      "E": 1645,
      "I": 690,
      "S": 1399
    },
    "eval": {
      "C": 548,
      "E": 2814,
      "I": 1032,
      "S": 2495
    }
  },
  "faltantes_%": {
    "descripcion": 47.9,
    "caracteristicas": 12.0,
    "marca": 4.7,
    "color": 37.2
  }
}


## Revisar separación y cobertura

In [8]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
consultas = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
qrels = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")

assert len(catalogo) >= C.MIN_CATALOGO
assert (consultas["conjunto"] == "eval").sum() >= 200
assert qrels["product_id"].isin(catalogo["product_id"]).all()
assert "esci_label" not in catalogo.columns

print("Catálogo:", len(catalogo))
print("Consultas dev/eval:", consultas["conjunto"].value_counts().to_dict())
print("Pares anotados:", len(qrels))
print("Etiquetas separadas del catálogo: OK")


Catálogo: 10688
Consultas dev/eval: {'eval': 250, 'dev': 150}
Pares anotados: 10928
Etiquetas separadas del catálogo: OK


## Batería de pruebas del catálogo

Estos cinco controles muestran qué se preparó y verifican que catálogo, consultas y etiquetas estén separados y listos para recuperación y evaluación.

In [9]:
filas_pruebas_datos = []

def agregar_prueba_datos(nombre, esperado, real, ok):
    filas_pruebas_datos.append({"Prueba": nombre, "Esperado": esperado,
                               "Resultado real": real, "Estado": "OK" if ok else "REVISAR"})

ids_dev = set(consultas.loc[consultas.conjunto == "dev", "query_id"])
ids_eval = set(consultas.loc[consultas.conjunto == "eval", "query_id"])
agregar_prueba_datos(
    "1. Tamaño del catálogo", f"al menos {C.MIN_CATALOGO:,} productos",
    f"{len(catalogo):,} productos", len(catalogo) >= C.MIN_CATALOGO)
agregar_prueba_datos(
    "2. Consultas de desarrollo y evaluación", "ambos grupos tienen consultas distintas",
    f"dev={len(ids_dev)}, eval={len(ids_eval)}, intersección={len(ids_dev & ids_eval)}",
    bool(ids_dev) and bool(ids_eval) and ids_dev.isdisjoint(ids_eval))
agregar_prueba_datos(
    "3. Cobertura de evaluación", f"al menos {C.N_CONSULTAS_EVAL} consultas eval",
    f"{len(ids_eval)} consultas eval", len(ids_eval) >= C.N_CONSULTAS_EVAL)
agregar_prueba_datos(
    "4. Etiquetas fuera del catálogo", "el catálogo no contiene esci_label",
    f"columnas del catálogo: {list(catalogo.columns)}", "esci_label" not in catalogo.columns)
ids_catalogo = set(catalogo.product_id)
agregar_prueba_datos(
    "5. Productos anotados presentes", "todos los qrels apuntan al catálogo",
    f"{qrels.product_id.isin(ids_catalogo).sum():,}/{len(qrels):,} etiquetas con producto presente",
    qrels.product_id.isin(ids_catalogo).all())

tabla_pruebas_datos = pd.DataFrame(filas_pruebas_datos)
display(tabla_pruebas_datos)

,Prueba,Esperado,Resultado real,Estado
0,1. Tamaño del catálogo,"al menos 5,000 productos","10,688 productos",OK
1,2. Consultas de desarrollo y evaluación,ambos grupos tienen consultas distintas,"dev=150, eval=250, intersección=0",OK
2,3. Cobertura de evaluación,al menos 250 consultas eval,250 consultas eval,OK
3,4. Etiquetas fuera del catálogo,el catálogo no contiene esci_label,"columnas del catálogo: ['product_id', 'titulo'...",OK
4,5. Productos anotados presentes,todos los qrels apuntan al catálogo,"10,928/10,928 etiquetas con producto presente",OK
